### Set explicit data directory and load pyspark

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("data-modeling-inmon")
    .getOrCreate()
)

print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 21:31:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 21:31:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3.5.4


## Inmon Model

Entities:
- Customer
- Order
- Order Item
- Product

### Creating silver customer

In [3]:
bronze_customers = spark.read.parquet(
    str(DATA_DIR / "bronze" / "crm_customers")
)

bronze_customers.show()

+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_id|first_name|last_name|               email|       city|state|         created_at|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+



In [4]:
inmon_customer = bronze_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "city",
    "state",
    "created_at"
)

inmon_customer.show()

+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_id|first_name|last_name|               email|       city|state|         created_at|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+



In [5]:
inmon_customer.write.mode("overwrite").parquet(
    str(DATA_DIR / "inmon" / "customer")
)

In [6]:
spark.read.parquet(
    str(DATA_DIR / "inmon" / "customer")
).show()

+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_id|first_name|last_name|               email|       city|state|         created_at|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+



### Creating orders silver

In [7]:
bronze_orders = spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_orders")
)

bronze_orders.show()

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|order_status|total_amount|
+--------+-----------+----------+------------+------------+
|    5001|       1001|2026-03-10|   Completed|      129.99|
|    5002|       1002|2026-03-11|   Completed|       84.50|
|    5003|       1001|2026-03-14|     Pending|       42.00|
|    5004|       1004|2026-03-18|   Completed|      215.75|
|    5005|       1003|2026-03-20|   Cancelled|       59.99|
+--------+-----------+----------+------------+------------+



In [8]:
inmon_order = bronze_orders.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_status",
    "total_amount"
)

inmon_order.show()

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|order_status|total_amount|
+--------+-----------+----------+------------+------------+
|    5001|       1001|2026-03-10|   Completed|      129.99|
|    5002|       1002|2026-03-11|   Completed|       84.50|
|    5003|       1001|2026-03-14|     Pending|       42.00|
|    5004|       1004|2026-03-18|   Completed|      215.75|
|    5005|       1003|2026-03-20|   Cancelled|       59.99|
+--------+-----------+----------+------------+------------+



In [9]:
inmon_order.write.mode("overwrite").parquet(
    str(DATA_DIR / "inmon" / "order")
)

In [10]:
spark.read.parquet(
    str(DATA_DIR / "inmon" / "order")
).show()

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|order_status|total_amount|
+--------+-----------+----------+------------+------------+
|    5001|       1001|2026-03-10|   Completed|      129.99|
|    5002|       1002|2026-03-11|   Completed|       84.50|
|    5003|       1001|2026-03-14|     Pending|       42.00|
|    5004|       1004|2026-03-18|   Completed|      215.75|
|    5005|       1003|2026-03-20|   Cancelled|       59.99|
+--------+-----------+----------+------------+------------+



### Write order items silver

In [11]:
bronze_order_items = spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_order_items")
)

bronze_order_items.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|    5001|      2001|       1|     34.99|
|    5001|      2002|       1|     89.99|
|    5002|      2003|       2|     44.99|
|    5003|      2004|       1|     54.99|
|    5004|      2002|       2|     89.99|
|    5005|      2001|       1|     34.99|
+--------+----------+--------+----------+



In [12]:
inmon_order_item = bronze_order_items.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
)

inmon_order_item.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|    5001|      2001|       1|     34.99|
|    5001|      2002|       1|     89.99|
|    5002|      2003|       2|     44.99|
|    5003|      2004|       1|     54.99|
|    5004|      2002|       2|     89.99|
|    5005|      2001|       1|     34.99|
+--------+----------+--------+----------+



In [13]:
inmon_order_item.write.mode("overwrite").parquet(
    str(DATA_DIR / "inmon" / "order_item")
)

In [14]:
spark.read.parquet(
    str(DATA_DIR / "inmon" / "order_item")
).show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|    5001|      2001|       1|     34.99|
|    5001|      2002|       1|     89.99|
|    5002|      2003|       2|     44.99|
|    5003|      2004|       1|     54.99|
|    5004|      2002|       2|     89.99|
|    5005|      2001|       1|     34.99|
+--------+----------+--------+----------+



### Write products silver

In [15]:
bronze_products = spark.read.parquet(
    str(DATA_DIR / "bronze" / "erp_products")
)

bronze_products.show()

+----------+-------------------+-----------+---------+----------+
|product_id|       product_name|   category|unit_cost|unit_price|
+----------+-------------------+-----------+---------+----------+
|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+----------+-------------------+-----------+---------+----------+



In [16]:
inmon_product = bronze_products.select(
    "product_id",
    "product_name",
    "category",
    "unit_cost",
    "unit_price"
)

inmon_product.show()

+----------+-------------------+-----------+---------+----------+
|product_id|       product_name|   category|unit_cost|unit_price|
+----------+-------------------+-----------+---------+----------+
|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+----------+-------------------+-----------+---------+----------+



In [17]:
inmon_product.write.mode("overwrite").parquet(
    str(DATA_DIR / "inmon" / "product")
)

In [18]:
spark.read.parquet(
    str(DATA_DIR / "inmon" / "product")
).show()

+----------+-------------------+-----------+---------+----------+
|product_id|       product_name|   category|unit_cost|unit_price|
+----------+-------------------+-----------+---------+----------+
|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+----------+-------------------+-----------+---------+----------+



Our flow is:
Source systems → Bronze → normalized enterprise model <br>
Later, if we build a reporting mart from this, that would look more like:
Inmon EDW → dimensional data mart